# Demo: Context windows, sub-agents, and research (Tavily + Exa + `langgraph-supervisor`)

This notebook shows **context engineering** for research: you split **Tavily** (broad web search) and **Exa** (semantic / neural retrieval) across **specialized workers**, then compare how the **supervisor** passes history back into the shared transcript.

## What you will see

1. **Prebuilt supervisor** — `create_supervisor` with **default handoff tools** and `output_mode="full_history"` (every worker turn stays in the global message list; easy to debug, heavier context).
2. **Customized supervisor** — explicit `create_handoff_tool` descriptions plus `output_mode="last_message"`, `add_handoff_messages=False`, and `create_forward_message_tool` so the supervisor can **promote a worker reply without re-paraphrasing** ([handoff customization](https://reference.langchain.com/python/langgraph-supervisor/supervisor/create_supervisor)).
3. **Skills-style single agent** — one ReAct agent with a `load_research_skill` tool (**progressive disclosure** of prompts) and **both** search tools, as in the [Skills pattern](https://docs.langchain.com/oss/python/langchain/multi-agent/skills) (fewer delegation hops; all tools stay in the same tool list).
4. **Checkpointed multi-turn supervisor** — same `langgraph-supervisor` graph compiled with [`InMemorySaver`](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.memory.InMemorySaver) so a **thread id** carries state across user turns ([supervisor + memory](https://reference.langchain.com/python/langgraph-supervisor)).
5. **Custom `Command` handoffs + structured delegation** — handoff tools append a **`HumanMessage`** that carries the supervisor’s focused sub-task (same idea as a `task_description` field, but **only `messages`** touch the parent graph). This avoids LangGraph 1.x + `langgraph-supervisor` issues where **`remaining_steps`** updates from the supervisor subgraph don’t match the parent schema and get dropped (warnings + broken / endless loops).
6. **LangChain supervisor-as-tools** — `langchain.agents.create_agent` with **only** high-level tools that wrap `tavily_agent` / `exa_agent` `.invoke`, matching the current [supervisor tutorial](https://docs.langchain.com/oss/python/langchain/supervisor).

## Why this matters

- **Multi-agent** is often really about **keeping the right context in the right place** ([multi-agent overview](https://docs.langchain.com/oss/python/langchain/multi-agent)): isolated workers avoid stuffing one prompt with every retrieval style’s instructions and raw results.
- **Handoffs vs skills**: handoffs isolate tools and prompts per subgraph; skills keep **one** controller and load **specialized instructions** on demand—different tradeoffs for **model calls**, **tokens**, and **parallelism** (see the performance discussion in the same multi-agent doc).

> **Note:** This notebook uses **`langgraph-supervisor`** to teach **history modes** and **handoff** mechanics, then ends with the **`create_agent` supervisor-as-tools** style LangChain recommends for many new apps ([multi-agent guide](https://docs.langchain.com/oss/python/langchain/multi-agent)).

## References

- [`langgraph-supervisor` reference](https://reference.langchain.com/python/langgraph-supervisor)
- [Context engineering in agents](https://docs.langchain.com/oss/python/langchain/context-engineering)
- [Tavily](https://tavily.com/) · [Exa](https://exa.ai/)

## Prerequisites

- `OPENAI_API_KEY` (or set `OPENAI_BASE_URL` + key for another OpenAI-compatible endpoint)
- `TAVILY_API_KEY` — [Tavily dashboard](https://app.tavily.com)
- `EXA_API_KEY` — [Exa dashboard](https://dashboard.exa.ai)

Optional: `.env` in the project root. **Restart the kernel** after the first `pip install` if imports fail.

**LangGraph 1.x note:** `create_supervisor`’s outer graph state is **`messages` only**. Custom `state_schema` that adds `remaining_steps` often still won’t register that managed channel on the **parent**, so the supervisor subgraph’s step counter updates are ignored—spam warnings and unstable loops. This notebook keeps the **default** parent schema and puts delegation text **in the message list** instead.

In [4]:
%pip install -q -U \
    langgraph \
    langchain \
    langchain-openai \
    langgraph-supervisor \
    langchain-tavily \
    exa_py \
    python-dotenv \
    tiktoken

In [5]:
import os

from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("OPENAI_API_KEY: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if not TAVILY_API_KEY:
    TAVILY_API_KEY = input("TAVILY_API_KEY: ").strip()
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
if not EXA_API_KEY:
    EXA_API_KEY = input("EXA_API_KEY: ").strip()
    os.environ["EXA_API_KEY"] = EXA_API_KEY

assert OPENAI_API_KEY and TAVILY_API_KEY and EXA_API_KEY, "All three keys are required."

# Optional: OpenAI-compatible base URL (e.g. proxy)
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")

print("Model:", OPENAI_MODEL)

Model: gpt-5.4-nano


In [6]:
from __future__ import annotations

import json
from typing import Annotated, Any

import tiktoken
from exa_py import Exa
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import BaseTool, InjectedToolCallId, tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langgraph_supervisor import create_supervisor, create_handoff_tool
from langgraph_supervisor.handoff import METADATA_KEY_HANDOFF_DESTINATION, create_forward_message_tool


def approx_tokens_from_messages(messages: list[BaseMessage], encoding_name: str = "cl100k_base") -> int:
    """Rough token count for the serialized supervisor transcript (teaching metric, not billing)."""
    enc = tiktoken.get_encoding(encoding_name)
    total = 0
    for m in messages:
        total += len(enc.encode(m.content if isinstance(m.content, str) else json.dumps(m.content)))
        total += len(enc.encode(getattr(m, "type", m.__class__.__name__)))
    return total


def build_model() -> ChatOpenAI:
    kwargs: dict[str, Any] = {"model": OPENAI_MODEL, "temperature": 0}
    if OPENAI_BASE_URL:
        kwargs["base_url"] = OPENAI_BASE_URL
    return ChatOpenAI(**kwargs)


# Caps runaway graphs if the model keeps calling tools (especially when debugging supervisor loops).
RUN_CFG = {"recursion_limit": 60}

model = build_model()

tavily_lc = TavilySearch(max_results=4, search_depth="advanced")
tavily_tool = tavily_lc.as_tool()


@tool("exa_neural_search")
def exa_neural_search(query: str) -> str:
    """Semantic / neural web search with Exa; returns titles, URLs, and highlight snippets."""
    exa = Exa(api_key=EXA_API_KEY)
    response = exa.search_and_contents(
        query,
        type="neural",
        num_results=4,
        highlights=True,
    )
    rows = []
    for r in response.results:
        hl = "".join(r.highlights) if r.highlights else ""
        rows.append({"title": r.title, "url": r.url, "highlights": hl})
    return json.dumps(rows, ensure_ascii=False)


def create_task_handoff_tool(*, agent_name: str, name: str, description: str) -> BaseTool:
    """`Command` handoff that injects the delegated sub-task as a `HumanMessage` (transcript-only).

    LangGraph 1.x + `create_supervisor` keeps the **parent** state as `messages` only. Writing `task_description`
    or merging `**state` can fight that contract. Specialists see the delegation in `messages` like any other turn.
    """

    @tool(name, description=description)
    def handoff_with_task(
        task_description: Annotated[
            str,
            "Focused sub-task for this specialist: search angles, scope, and desired output shape.",
        ],
        state: Annotated[dict, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        msgs = state["messages"]
        if msgs and isinstance(msgs[-1], AIMessage):
            forward_msgs = msgs[:-1]
        else:
            forward_msgs = list(msgs)
        delegation = HumanMessage(
            content=(
                f"[Supervisor → {agent_name}] Focused sub-task:\n{task_description}"
            )
        )
        return Command(
            goto=agent_name,
            graph=Command.PARENT,
            update={"messages": forward_msgs + [delegation]},
        )

    handoff_with_task.metadata = {METADATA_KEY_HANDOFF_DESTINATION: agent_name}
    return handoff_with_task


SUPERVISOR_BRIEF = (
    "You coordinate two researchers: one uses Tavily (fresh web/news), one uses Exa (semantic retrieval). "
    "For each user question, delegate at least one search pass, then synthesize a short answer with bullet points and URLs. "
    "Prefer calling both specialists when the question benefits from different retrieval styles."
)


TAVILY_SYSTEM_PROMPT = (
    "You are a Tavily web researcher. Use the Tavily tool with focused queries. "
    "Return concise notes plus source URLs. Do not claim you used Exa. "
    "When the transcript includes a line starting with [Supervisor → tavily_researcher], treat that block as your priority task."
)

EXA_SYSTEM_PROMPT = (
    "You are an Exa semantic researcher. Use exa_neural_search with precise, semantic queries. "
    "Return concise notes plus URLs. Do not claim you used Tavily. "
    "When the transcript includes a line starting with [Supervisor → exa_researcher], treat that block as your priority task."
)

tavily_agent = create_agent(
    model,
    tools=[tavily_tool],
    system_prompt=TAVILY_SYSTEM_PROMPT,
    name="tavily_researcher",
)

exa_agent = create_agent(
    model,
    tools=[exa_neural_search],
    system_prompt=EXA_SYSTEM_PROMPT,
    name="exa_researcher",
)

/tmp/ipykernel_2586/1616203901.py:42: LangChainBetaWarning: This API is in beta and may change in the future.
  tavily_tool = tavily_lc.as_tool()


## 1) Prebuilt supervisor — default handoffs + full history

Default `create_supervisor` wiring keeps **worker tool traces** in the shared transcript (`output_mode="full_history"`). That is ideal for **debugging** and LangSmith traces, but it grows **token** usage quickly—exactly the tension described in LangChain’s [context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering) guide (what the next model call *sees* vs what you persist).

In [7]:
USER_TASK = (
    "Briefly compare how LangGraph multi-agent patterns are documented in 2025: "
    "pull one Tavily-oriented angle (news/blog/docs landing pages) and one Exa-oriented angle "
    "(semantic match on technical explanations). Then give 3 bullets + citations."
)

workflow_prebuilt = create_supervisor(
    [tavily_agent, exa_agent],
    model=model,
    prompt=SUPERVISOR_BRIEF,
    output_mode="full_history",
)

app_prebuilt = workflow_prebuilt.compile()
result_prebuilt = app_prebuilt.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    RUN_CFG,
)

msgs_pre = result_prebuilt["messages"]
print("Prebuilt — messages:", len(msgs_pre), "| ~tokens:", approx_tokens_from_messages(msgs_pre))
print("---")
print(msgs_pre[-1].content)

Prebuilt — messages: 22 | ~tokens: 12182
---
### Brief comparison (2025 documentation style)

- **Tavily-oriented (news/blog/docs landing pages):** In 2025, “LangGraph multi-agent patterns” are commonly documented as **build guidance**—end-to-end walkthroughs that emphasize *production concerns* (stateful branching, human-in-the-loop, iteration loops) and often showcase **supervisor/router**-like architectures as the practical default.  
  - https://www.digitalapplied.com/blog/langchain-ai-agents-guide-2025

- **Exa-oriented (semantic match on technical explanations):** The technical explanation side tends to treat “patterns” as **graph control-flow primitives**—specifically **state**, **routing functions**, and **conditional edges**—which semantically correspond to multi-agent strategies like supervisor/router and hierarchical teams.  
  - https://langchain-tutorials.github.io/langgraph-conditional-edges-router-pattern-guide/

### 3 bullets + citations

- **2025 “patterns” in blog/lan

## 2) Customized supervisor — handoff tools + last message + forward

Here we:

- Replace generic handoff names with **explicit** `create_handoff_tool` descriptions so the supervisor states *why* it is routing.
- Set `output_mode="last_message"` so only the **final worker message** merges back for each delegation round ([message history management](https://reference.langchain.com/python/langgraph-supervisor)).
- Set `add_handoff_messages=False` for a **slimmer** transcript.
- Add `create_forward_message_tool` so the supervisor can **forward** a worker’s answer as the user-visible result without rewriting it (saves tokens and avoids paraphrase drift).

Run the cell below and compare the printed **~tokens** figure to the prebuilt run.

In [8]:
handoff_tavily = create_handoff_tool(
    agent_name="tavily_researcher",
    name="assign_tavily_web_research",
    description=(
        "Delegate to the Tavily specialist for broad web search, recent pages, and documentation landing URLs. "
        "Use for newsy or shallow crawl-friendly queries."
    ),
)

handoff_exa = create_handoff_tool(
    agent_name="exa_researcher",
    name="assign_exa_semantic_research",
    description=(
        "Delegate to the Exa specialist for semantic retrieval and highlight snippets from technically similar pages. "
        "Use for conceptual / tutorial-style matches."
    ),
)

forward_tool = create_forward_message_tool("supervisor")

workflow_custom = create_supervisor(
    [tavily_agent, exa_agent],
    model=model,
    tools=[handoff_tavily, handoff_exa, forward_tool],
    prompt=(
        SUPERVISOR_BRIEF
        + " When a worker’s last message already answers the user, call forward_message with that worker’s name instead of rewriting."
    ),
    output_mode="last_message",
    add_handoff_messages=False,
)

app_custom = workflow_custom.compile()
result_custom = app_custom.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    RUN_CFG,
)

msgs_cust = result_custom["messages"]
print("Custom — messages:", len(msgs_cust), "| ~tokens:", approx_tokens_from_messages(msgs_cust))
print("---")
print(msgs_cust[-1].content)

Custom — messages: 8 | ~tokens: 1357
---
- **Tavily-oriented (landing/news/docs entry angle):** In LangChain’s 2025-style docs/blog framing, LangGraph multi-agent patterns are introduced primarily as **graph-based orchestration for long-running, stateful agent workflows** (explicit nodes/edges + durability), not as “conversation-style” agent collaboration.  
  https://docs.langchain.com/oss/python/langgraph/overview  
  https://blog.langchain.com/langgraph-multi-agent-workflows/

- **Exa-oriented (semantic/mechanism explanation angle):** The technical “multi-agent patterns” pages explain patterns as **specific control-flow + state/routing mechanisms**—e.g., how a **Router** dispatches to specialized paths/agents and how **Skills/Handoffs** change behavior based on state/tool outcomes.  
  https://docs.langchain.com/oss/python/langchain/multi-agent/router  
  https://docs.langchain.com/oss/python/langchain/multi-agent/skills  
  https://docs.langchain.com/oss/python/langchain/multi-agen

## 3) Skills pattern — one agent, progressive prompt disclosure

The [Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills) pattern keeps **one** agent in control and loads **specialized instructions** through a tool (`load_research_skill`). That mirrors progressive disclosure (similar in spirit to `llms.txt` / Agent Skills): you do not inject every playbook into the system prompt up front.

**Tradeoff:** both Tavily and Exa stay in the **same** tool list, so the model must choose tools without the hard isolation you get from separate sub-agents—often fewer **delegation** steps, sometimes more **tool-selection** errors on complex stacks.

In [9]:
SKILLS: dict[str, str] = {
    "tavily_web": (
        "Skill: Tavily web search.\n"
        "- Issue short, keyword-rich queries.\n"
        "- Prefer official docs or reputable blogs for release notes.\n"
        "- Summarize with URLs.\n"
    ),
    "exa_neural": (
        "Skill: Exa neural search.\n"
        "- Phrase queries as semantic questions (concept + technology).\n"
        "- Use highlights as evidence, not as full page text.\n"
        "- Summarize with URLs.\n"
    ),
}


@tool
def load_research_skill(skill_name: str) -> str:
    """Load specialized research instructions. Use before searching.

    Args:
        skill_name: "tavily_web" or "exa_neural".
    """
    return SKILLS.get(skill_name, f"Unknown skill {skill_name!r}. Use tavily_web or exa_neural.")


skills_agent = create_agent(
    model,
    tools=[tavily_tool, exa_neural_search, load_research_skill],
    system_prompt=(
        "You are a single research agent with two search backends. "
        "Before each distinct search strategy, call load_research_skill with the matching skill name. "
        "Then call the appropriate search tool. Finish with 3 bullets + URLs."
    ),
    name="skills_researcher",
)

result_skills = skills_agent.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    RUN_CFG,
)

msgs_sk = result_skills["messages"]
print("Skills agent — messages:", len(msgs_sk), "| ~tokens:", approx_tokens_from_messages(msgs_sk))
print("---")
print(msgs_sk[-1].content)

Skills agent — messages: 10 | ~tokens: 5283
---
- **Tavily (docs/blog landing-page angle):** In 2025, multi-agent patterns (e.g., supervisor + orchestration) are commonly described at the *feature/benefits level* on landing-style pages—emphasizing *stateful orchestration*, *interrupts/human-in-the-loop*, *durability*, and *debugging/tracing (LangSmith)*—rather than only deep control-flow details. Example: LangGraph’s official overview frames why graph-based multi-agent orchestration matters for complex, long-running workflows.  
  https://docs.langchain.com/oss/python/langgraph/overview

- **Exa (semantic technical-explanation angle):** Technical writeups found via semantic matching tend to explain the *actual topology mechanics* (e.g., **hub-and-spoke supervisor routing**, **conditional edges**, and how agents return control/state to decide the next node). These pages go beyond “what” into “how” (routing decisions based on state, re-invocation loops, merging outputs).  
  https://myen

## 4) Multi-turn supervisor with `InMemorySaver`

`create_supervisor` returns a `StateGraph`; pass a [checkpointer](https://langchain-ai.github.io/langgraph/concepts/persistence/) to `.compile()` so **messages** survive across `.invoke` calls when you reuse the same `thread_id` ([supervisor memory](https://reference.langchain.com/python/langgraph-supervisor)).

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
app_threaded = workflow_custom.compile(checkpointer=checkpointer)

thread_cfg = {**RUN_CFG, "configurable": {"thread_id": "research-demo-thread-1"}}

_ = app_threaded.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    thread_cfg,
)
followup = app_threaded.invoke(
    {"messages": [{"role": "user", "content": "Add one short sentence on API cost tradeoffs between these search styles."}]},
    thread_cfg,
)

msgs_thread = followup["messages"]
print("After follow-up — messages:", len(msgs_thread), "| ~tokens:", approx_tokens_from_messages(msgs_thread))
delegation_msgs = [m for m in msgs_thread if getattr(m, "content", "") and "[Supervisor →" in str(m.content)]
print("delegation HumanMessages in thread:", len(delegation_msgs))
print("---")
print(msgs_thread[-1].content)

After follow-up — messages: 10 | ~tokens: 985
delegation HumanMessages in thread: 0
---
- These search styles have different API cost tradeoffs: Tavily-style broader crawling/landing-page discovery can cost more per relevant hit, while Exa-style semantic retrieval can be cheaper when your query matches well to a specific technical concept (fewer results to open).


## 5) Custom `Command` handoffs with explicit delegation text

The supervisor must pass a rich **`task_description`** argument to **`delegate_tavily_with_task`** / **`delegate_exa_with_task`**. Those tools return a **`Command`** that routes to the worker and appends a **`HumanMessage`** (`[Supervisor → …]`) so the specialist sees a clear sub-goal **in the transcript**—no extra parent state keys. This mirrors the docs’ **custom `Command` handoff** pattern while staying compatible with LangGraph 1’s **messages-only** parent state from `create_supervisor` ([langgraph-supervisor](https://reference.langchain.com/python/langgraph-supervisor)).

In [11]:
task_handoff_tavily = create_task_handoff_tool(
    agent_name="tavily_researcher",
    name="delegate_tavily_with_task",
    description=(
        "Send a focused Tavily web search task. Populate task_description with queries, scope, and output shape."
    ),
)

task_handoff_exa = create_task_handoff_tool(
    agent_name="exa_researcher",
    name="delegate_exa_with_task",
    description=(
        "Send a focused Exa semantic search task. Populate task_description with the conceptual question and evidence needs."
    ),
)

forward_cmd = create_forward_message_tool("supervisor")

workflow_task_cmd = create_supervisor(
    [tavily_agent, exa_agent],
    model=model,
    tools=[task_handoff_tavily, task_handoff_exa, forward_cmd],
    prompt=(
        SUPERVISOR_BRIEF
        + " Whenever you delegate, call delegate_tavily_with_task or delegate_exa_with_task with a rich task_description. "
        "When a specialist's answer is sufficient, use forward_message to avoid rewriting."
    ),
    output_mode="last_message",
    add_handoff_messages=False,
)

app_task_cmd = workflow_task_cmd.compile()
result_task_cmd = app_task_cmd.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    RUN_CFG,
)

msgs_cmd = result_task_cmd["messages"]
print("Command handoffs — messages:", len(msgs_cmd), "| ~tokens:", approx_tokens_from_messages(msgs_cmd))
deleg = [m for m in msgs_cmd if getattr(m, "content", "") and "[Supervisor →" in str(m.content)]
print("delegation HumanMessages:", len(deleg))
print("---")
print(msgs_cmd[-1].content)

Command handoffs — messages: 6 | ~tokens: 1623
delegation HumanMessages: 2
---
### 3 bullets (compare documentation style in 2025, with citations)

- **Tavily angle (news/docs landing pages):** LangGraph multi-agent patterns are commonly introduced as *named patterns + tutorial pathways* (e.g., “hierarchical agent teams” and “multi-agent workflows” pattern collections). This framing emphasizes *what to build and in what order* rather than the internal graph/routing mechanics.  
  Citations: https://langchain-ai.github.io/langgraph/tutorials/multi_agent/hierarchical_agent_teams , https://blog.langchain.com/langgraph-multi-agent-workflows/

- **Exa angle (technical explanations via semantic match):** The most mechanism-heavy pages explain multi-agent patterns as *compiled graph/state machinery*—supervisors/router behavior is described via routing/edges, state schemas, handoff/control-transfer helpers, and output/streaming modes (e.g., what the supervisor returns).  
  Citations: https://

## 6) LangChain 1.x supervisor — subagents as tools only

The [supervisor tutorial](https://docs.langchain.com/oss/python/langchain/supervisor) wraps each specialist as a **tool** that runs `sub_agent.invoke(...)` and returns only the **final** assistant text. The outer `create_agent` never sees Tavily/Exa tools directly—only `research_via_tavily` and `research_via_exa`.

In [12]:
def _final_ai_text(invoke_result: dict) -> str:
    m = invoke_result["messages"][-1]
    t = getattr(m, "text", None)
    if t is not None:
        return t
    c = m.content
    return c if isinstance(c, str) else str(c)


@tool
def research_via_tavily(task: str) -> str:
    """High-level Tavily web search for fresh pages, news, and documentation URLs."""
    out = tavily_agent.invoke({"messages": [{"role": "user", "content": task}]}, RUN_CFG)
    return _final_ai_text(out)


@tool
def research_via_exa(task: str) -> str:
    """High-level Exa semantic / neural search for deep technical explanations."""
    out = exa_agent.invoke({"messages": [{"role": "user", "content": task}]}, RUN_CFG)
    return _final_ai_text(out)


SUPERVISOR_TOOLS_PROMPT = (
    "You supervise research. Use research_via_tavily for broad web discovery and landing pages; "
    "use research_via_exa for semantic technical angles. Combine into 3 bullets with URLs."
)

supervisor_tools_agent = create_agent(
    model,
    tools=[research_via_tavily, research_via_exa],
    system_prompt=SUPERVISOR_TOOLS_PROMPT,
)

result_lc = supervisor_tools_agent.invoke(
    {"messages": [{"role": "user", "content": USER_TASK}]},
    RUN_CFG,
)
msgs_lc = result_lc["messages"]
print("Supervisor-as-tools — messages:", len(msgs_lc), "| ~tokens:", approx_tokens_from_messages(msgs_lc))
print("---")
print(_final_ai_text(result_lc))

Supervisor-as-tools — messages: 5 | ~tokens: 1593
---
- **Tavily (2025 blog/docs landing-page angle):** In 2025-style documentation, multi-agent patterns are often presented as **graph-based orchestration** (nodes/edges for agents, state handled by the graph runtime) and **hierarchical supervision** (a supervisor delegates to sub-agents/“workers” and then aggregates results). Example landing/doc pages: LangGraph “overview” (positioning/orchestration) and supervisor/sub-agent docs/pages.  
  https://docs.langchain.com/oss/python/langgraph/overview  
  https://reference.langchain.com/python/langgraph-supervisor

- **Exa (semantic/technical-explanations angle):** Technical explanations tend to emphasize **state semantics** and **handoff mechanics**: how the shared graph state is defined/merged (reducers), how agent updates (especially **messages**) are appended/combined, and how supervisor routing connects conditional edges to worker outputs. Key technical anchors include persistence/chec

## 7) Side-by-side metrics (illustrative)

Numbers will **vary** by model version, search results, and routing. Qualitative guide:

| Approach | Typical story |
|----------|----------------|
| Prebuilt + `full_history` | Richest trace; **highest** token footprint. |
| Custom + `last_message` + `forward_message` | Slimmer merged transcript. |
| Skills single agent | One tool surface; no supervisor graph. |
| Threaded + checkpointer | Same as custom after **two** user turns—history grows with thread. |
| `Command` handoffs + `HumanMessage` | Forces explicit sub-goals; workers see **`[Supervisor → …]`** lines in **messages**. |
| `create_agent` supervisor tools | Outer agent sees **only** tool summaries—often compact at supervisor layer. |

See [multi-agent patterns](https://docs.langchain.com/oss/python/langchain/multi-agent) for when to prefer isolation vs skills vs tool-wrapped subagents.

In [13]:
summary = [
    ("prebuilt_full_history", len(msgs_pre), approx_tokens_from_messages(msgs_pre)),
    ("custom_last_message_forward", len(msgs_cust), approx_tokens_from_messages(msgs_cust)),
    ("skills_single_agent", len(msgs_sk), approx_tokens_from_messages(msgs_sk)),
    ("threaded_two_turns", len(msgs_thread), approx_tokens_from_messages(msgs_thread)),
    ("task_command_handoffs", len(msgs_cmd), approx_tokens_from_messages(msgs_cmd)),
    ("langchain_supervisor_tools", len(msgs_lc), approx_tokens_from_messages(msgs_lc)),
]

for name, n_msg, tok in summary:
    print(f"{name:32} messages={n_msg:3}  ~tokens={tok:6}")

print("\nLowest ~tokens this run:", min(summary, key=lambda x: x[2])[0])

prebuilt_full_history            messages= 22  ~tokens= 12182
custom_last_message_forward      messages=  8  ~tokens=  1357
skills_single_agent              messages= 10  ~tokens=  5283
threaded_two_turns               messages= 10  ~tokens=   985
task_command_handoffs            messages=  6  ~tokens=  1623
langchain_supervisor_tools       messages=  5  ~tokens=  1593

Lowest ~tokens this run: threaded_two_turns
